# Desafio Técnico — Data Scientist | acaso

## Objetivo
Mapear skills extraídas de perfis de colaboradores para uma taxonomia oficial de skills, utilizando uma abordagem automatizada baseada em similaridade semântica.

## Estratégia
A solução utiliza embeddings de texto para representar semanticamente tanto as skills brutas quanto a taxonomia.

A similaridade entre esses vetores é calculada utilizando cosine similarity, permitindo identificar, para cada skill, a entrada mais próxima na taxonomia.

Com base no score de similaridade, as skills são classificadas como:
- `matched` (quando o score ultrapassa o threshold definido)
- `no_match` (quando o score está abaixo do threshold)

A abordagem prioriza simplicidade, interpretabilidade e facilidade de ajuste, sendo adequada como uma primeira versão do sistema.

# Import libraries

In [245]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from pathlib import Path
import csv

# Carregamento  a  leitura dos arquivos CSV

In [247]:
DATA= Path("data")

new_skills_path = DATA / "new_skills.csv"
taxonomy_path = DATA / "skill_taxonomy.csv"

df_new = pd.read_csv(new_skills_path)
df_tax = pd.read_csv(taxonomy_path)

print("new_skills shape:", df_new.shape)
print("skill_taxonomy shape:", df_tax.shape)

new_skills shape: (3030, 2)
skill_taxonomy shape: (100, 3)


In [248]:
print("Total de skills na taxonomia:", len(df_tax))

Total de skills na taxonomia: 100


# ## 📊 Análise Exploratória dos Dados (EDA)
Nesta etapa, realizo uma análise inicial dos dados para entender sua estrutura, tipos de dados e possíveis inconsistências.

O objetivo é identificar:
- estrutura dos arquivos
- tipos de colunas
- presença de valores nulos
- padrões iniciais nos dados

Essa etapa é importante para garantir que os dados estejam adequados para o processamento e modelagem posteriores.

In [243]:
df_new = pd.read_csv('data/new_skills.csv')
df_new

,id,skill_raw
0,1,gestão de pessoas
1,2,People Management
2,3,SQL
3,4,sql avançado
4,5,machine learning
...,...,...
3025,3026,comunicar dados
3026,3027,desenvolvimento de APIs
3027,3028,pipeline orchestration
3028,3029,next.js


In [ ]:
df_tax = pd.read_csv('data/skill_taxonomy.csv')
df_tax

In [ ]:
df_tax = df_tax.head(30)
df_tax

In [ ]:
print(df_tax.info())
print()

In [ ]:
print("\nNulos em skill_taxonomy:")
print(df_tax.isna().sum())

In [ ]:
print("Top 10 habilidades repetidas (sem limpeza):")
print(df_tax['skill_name'].value_counts().head(10))

In [ ]:
print(df_new.info())
print()

In [ ]:
print("\nNulos em new_skills:")
print(df_new.isna().sum())

In [ ]:
# df_tax = df_tax.head(30)  # usado apenas para testes locais

In [ ]:
print("Top 10 habilidades repetidas (sem limpeza):")
print(df_new['skill_raw'].value_counts().head(10))

## ## 🧹 Limpeza e Padronização dos Dados

Nesta etapa, realizo a limpeza dos textos das skills para garantir consistência antes do processamento semântico.

As transformações aplicadas incluem:
- conversão para minúsculas
- remoção de espaços extras
- remoção de caracteres especiais

O objetivo é reduzir variações desnecessárias nos dados (como "Python", "PYTHON", "python ") e melhorar a qualidade das representações vetoriais geradas posteriormente.

In [ ]:
df_new['skill_clean'] = (
    df_new['skill_raw']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)
print(df_new['skill_clean'].head())

In [ ]:
print("Conferindo a limpeza em  linhas:")
display(df_new[['skill_raw', 'skill_clean']].head(20))

In [ ]:
print("Conferindo a limpeza em linhas aleatórias:")
display(df_new[['skill_raw', 'skill_clean']].sample(20))

#  🔁 Análise de Redundância nas Skills

Nesta etapa, analiso a presença de duplicidades nas skills após o processo de limpeza.

Foi identificada uma alta redundância nos dados, o que indica que diferentes registros podem representar a mesma habilidade, ainda que escritos de formas variadas.

No entanto, **não foram removidas neste momento**, pois cada registro pode representar um colaborador diferente ou um contexto distinto.

A decisão foi manter os dados completos e permitir que o modelo semântico (embeddings + similaridade) seja responsável por identificar e agrupar habilidades equivalentes.

Essa abordagem evita perda de informação e garante maior fidelidade ao cenário real dos dados.

In [161]:
total_duplicadas = df_new['skill_clean'].duplicated().sum()
total_unicas = df_new['skill_clean'].nunique()

print(f"Linhas repetidas encontradas: {total_duplicadas}")
print(f"Habilidades únicas que sobrarão: {total_unicas}")

Linhas repetidas encontradas: 1555
Habilidades únicas que sobrarão: 1475


# Limpando o dataset df_tax

In [230]:
df_tax['skill_name_clean'] = (
    df_tax['skill_name']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

display(df_tax[['skill_name', 'skill_name_clean', 'description']].head(10))

,skill_name,skill_name_clean,description
0,Gestão de Pessoas,gestão de pessoas,"Capacidade de liderar, desenvolver e engajar e..."
1,Comunicação Assertiva,comunicação assertiva,Habilidade de expressar ideias e feedbacks de ...
2,Resolução de Conflitos,resolução de conflitos,Capacidade de mediar e resolver divergências e...
3,Inteligência Emocional,inteligência emocional,"Capacidade de reconhecer, compreender e gerenc..."
4,Negociação,negociação,Habilidade de conduzir negociações de forma es...
5,Pensamento Crítico,pensamento crítico,Capacidade de analisar informações de forma si...
6,Resolução de Problemas,resolução de problemas,Habilidade de identificar causas raiz de probl...
7,Tomada de Decisão,tomada de decisão,Capacidade de avaliar alternativas e tomar dec...
8,Adaptabilidade,adaptabilidade,Capacidade de ajustar comportamentos e abordag...
9,Gestão do Tempo,gestão do tempo,Habilidade de organizar e priorizar tarefas de...


## Concatenei o nome da skill com sua descrição para enriquecer o contexto semântico antes de gerar os embeddings.
 
 Essa abordagem permite que o modelo capture não apenas o termo isolado, mas também seu significado e aplicação, melhorando a qualidade das representações vetoriais e aumentando a precisão dos matches.

 Exemplo: antes = "Python"
 depois = "Python - Linguagem de programação versátil"

In [231]:
df_tax['taxonomy_text'] = (
    df_tax['skill_name'] + ' - ' + df_tax['description']
)

display(df_tax[['skill_name', 'taxonomy_text']].head(10))

,skill_name,taxonomy_text
0,Gestão de Pessoas,"Gestão de Pessoas - Capacidade de liderar, des..."
1,Comunicação Assertiva,Comunicação Assertiva - Habilidade de expressa...
2,Resolução de Conflitos,Resolução de Conflitos - Capacidade de mediar ...
3,Inteligência Emocional,Inteligência Emocional - Capacidade de reconhe...
4,Negociação,Negociação - Habilidade de conduzir negociaçõe...
5,Pensamento Crítico,Pensamento Crítico - Capacidade de analisar in...
6,Resolução de Problemas,Resolução de Problemas - Habilidade de identif...
7,Tomada de Decisão,Tomada de Decisão - Capacidade de avaliar alte...
8,Adaptabilidade,Adaptabilidade - Capacidade de ajustar comport...
9,Gestão do Tempo,Gestão do Tempo - Habilidade de organizar e pr...


## 📊 Embeddings

Nesta etapa, transformei os textos em vetores numéricos utilizando um modelo de linguagem.

Isso permite capturar similaridade semântica entre as skills, indo além de comparações textuais diretas.

Foi utilizado o modelo `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, que é um modelo leve e otimizado para gerar embeddings semânticos.

Esse modelo suporta múltiplos idiomas (incluindo português e inglês), o que é essencial para este problema, já que as skills podem aparecer em diferentes línguas.

Além disso, ele oferece um bom equilíbrio entre performance e custo computacional, sendo adequado para processamento em larga escala.

In [232]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)

print("Modelo carregado:", model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo carregado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [233]:
taxonomy_texts = df_tax['taxonomy_text'].tolist()

taxonomy_embeddings = model.encode(
    taxonomy_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Shape embeddings taxonomia:", taxonomy_embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Shape embeddings taxonomia: (100, 384)


## 🔄 Geração de Embeddings das Skills Brutas

Nesta etapa, transformo as skills normalizadas (`skill_clean`) em vetores numéricos utilizando um modelo de linguagem.

Cada texto é convertido em um embedding, que representa seu significado semântico em um espaço vetorial.

Isso permite que habilidades com significados semelhantes, mesmo escritas de formas diferentes, sejam posicionadas próximas no espaço vetorial.

Esses vetores serão utilizados posteriormente para calcular similaridade com a taxonomia e encontrar o melhor match para cada skill.

In [234]:
new_skill_texts = df_new['skill_clean'].tolist()

new_skill_embeddings = model.encode( #transforma cada texto em vetor
    new_skill_texts, # Lista de textos
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Shape embeddings skills:", new_skill_embeddings.shape)

Batches:   0%|          | 0/95 [00:00<?, ?it/s]

Shape embeddings skills: (3030, 384)


## Cálculo de Similaridade

Aqui utilizamos cosine similarity para medir o quão próximos semanticamente estão os vetores das skills.

O objetivo é encontrar, para cada skill nova, a skill mais próxima na taxonomia.

In [235]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(
    new_skill_embeddings, #instacio embeddings das skills novas
    taxonomy_embeddings #instacio embeddings das skills da taxonomia
)

print("Shape matriz de similaridade:", similarity_matrix.shape)

Shape matriz de similaridade: (3030, 100)


## 🎯 Identificação do Melhor Match

Após calcular a matriz de similaridade entre as skills novas e a taxonomia, é necessário identificar, para cada skill, qual é a mais próxima semanticamente.

Para isso:

- Utilizo `argmax` para encontrar o índice da skill da taxonomia com maior similaridade para cada skill nova
- Utilizo `max` para obter o valor dessa similaridade, que será usado como score de confiança

Esse processo garante que cada skill seja associada ao melhor candidato disponível na taxonomia.

In [236]:
import numpy as np

best_match_indices = np.argmax(similarity_matrix, axis=1) # PARA CADA LINHA PEGA INDICE DA COLUNA COM MAIOR VALOR 

best_scores = np.max(similarity_matrix, axis=1) # pega o score desse maior valor

print("Exemplo índices:", best_match_indices[:10])
print("Exemplo scores:", best_scores[:10])

Exemplo índices: [ 0  0 31 31 39  1 30 30 38  2]
Exemplo scores: [0.7005039  0.7338302  0.7017696  0.72010183 0.80590385 0.7490694
 0.6242019  0.6242019  0.8053319  0.7446522 ]


# skill da taxonomia match

In [242]:
df_tax.iloc[39]['skill_name']

'Machine Learning'

## 📊 Consolidação do Resultado Final

Nesta etapa, consolido todas as informações geradas ao longo do pipeline em uma única tabela final.

Aqui, cada skill bruta é associada à melhor correspondência encontrada na taxonomia, juntamente com:
- o identificador da skill na taxonomia
- o nome padronizado da skill
- o score de similaridade (confidence_score)
- o status do match (matched ou no_match)

Essa estrutura permite visualizar de forma clara e organizada o resultado do mapeamento, facilitando análises posteriores e validação da qualidade da solução.

In [244]:
df_result = df_new.copy()

df_result['taxonomy_id'] = best_match_indices
df_result['confidence_score'] = best_scores

df_result['skill_name'] = df_result['taxonomy_id'].apply(
    lambda x: df_tax.iloc[x]['skill_name']
)

df_result['match_status'] = df_result['confidence_score'].apply(
    lambda x: 'matched' if x > 0.7 else 'no_match'
)

df_result = df_result[[
    'id',
    'skill_raw',
    'taxonomy_id',
    'skill_name',
    'confidence_score',
    'match_status'
]]

df_result.sample(20)

,id,skill_raw,taxonomy_id,skill_name,confidence_score,match_status
2890,2891,building pipelines,52,Orquestração de Pipelines,0.575343,no_match
2747,2748,customer needs understanding,15,Foco no Cliente,0.765389,matched
1207,1208,PO,89,OKRs,0.274575,no_match
1474,1475,looker,78,Looker,0.364471,no_match
307,308,avaliação de LLMs em produção,45,Avaliação de Modelos,0.580317,no_match
1485,1486,spreadsheet,77,Excel Avançado,0.377814,no_match
1745,1746,upskilling,89,OKRs,0.337689,no_match
507,508,KPIs,85,Hugging Face,0.316727,no_match
320,321,context switching,8,Adaptabilidade,0.493984,no_match
2003,2004,github,71,Versionamento de Código,0.730145,matched


# salvar resultado final


In [239]:
# salvar resultado final
df_result.to_csv("output_mapping.csv", index=False)  

print("Arquivo salvo com sucesso!")

Arquivo salvo com sucesso!
